In [3]:
#导入库
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
#设置路径并读取数据
# 当前 notebook 所在目录
PROJECT_DIR = Path.cwd()

# 文件就在当前目录
RAW_FILE = PROJECT_DIR / "Raw_data_work.xlsx"

print("当前目录:", PROJECT_DIR)
print("文件是否存在:", RAW_FILE.exists())

df = pd.read_excel(RAW_FILE, sheet_name="Raw_data")

print("数据形状:", df.shape)
df.head()

当前目录: D:\pythonfiles\Github_Projects\adwords_analysis_test
文件是否存在: True
数据形状: (199, 22)


,Title,Keyword,Keyword_Clean,Dup_Keyword,Position,Check_Position,Previous position,Search Volume,CPC,Traffic,...,Traffic Cost,Traffic Cost (%),Competition,Check_Competition,Number of Results,Trends,Last Seen,Check_Last Seen,Keyword Difficulty,Check_Keyword Difficulty
0,PMP® Exam & Certification | Flat 40% Off - Enr...,pmp certification,pmp certification,唯一,1,正常,1,90500,3.76,4253,...,15991,24.66,0.60,正常,166000000,"[67,54,81,81,67,67,81,67,81,81,67,67]",2024-12-31,日期正确,78.0,正常
1,PMP® Exam & Certification | Confidently Ace Th...,pmp certification,pmp certification,唯一,1,正常,1,90500,3.76,4253,...,15991,24.66,0.60,正常,172000000,"[67,54,81,81,67,67,81,67,81,81,67,67]",2025-01-01,日期正确,78.0,正常
2,PMP® Training & Exam | Special Offer: Get $400...,pmp certification,pmp certification,唯一,1,正常,2,90500,3.82,1176,...,4492,6.92,0.65,正常,154000000,"[54,81,81,67,67,81,67,81,81,67,67,81]",2025-01-14,日期正确,77.0,正常
3,CSM Scrum Master Course Online | Today's Offer...,scrum master certification,scrum master certification,唯一,1,正常,1,18100,3.21,850,...,2728,4.20,0.64,正常,32000000,"[44,54,54,44,44,44,54,54,44,54,44,44]",2025-01-10,日期正确,70.0,正常
4,PMP® Certification | Ace PMP Exam in 1st Attempt,pmp certification,pmp certification,唯一,3,正常,3,90500,3.82,814,...,3109,4.79,0.65,正常,165000000,"[54,81,81,67,67,81,67,81,81,67,67,81]",2025-01-07,日期正确,78.0,正常


In [5]:
#数据体检
print("\n=== 数据类型 ===")
print(df.info())

print("\n=== 缺失值统计 ===")
print(df.isna().sum().sort_values(ascending=False))

print("\n=== 重复行数量 ===")
print(df.duplicated().sum())


=== 数据类型 ===
<class 'pandas.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Title                     199 non-null    str           
 1   Keyword                   199 non-null    str           
 2   Keyword_Clean             199 non-null    str           
 3   Dup_Keyword               199 non-null    str           
 4   Position                  199 non-null    int64         
 5   Check_Position            199 non-null    str           
 6   Previous position         199 non-null    int64         
 7   Search Volume             199 non-null    int64         
 8   CPC                       199 non-null    float64       
 9   Traffic                   199 non-null    int64         
 10  Traffic (%)               199 non-null    float64       
 11  Check_Traffic (%)         199 non-null    str           
 12  Traffic Cost       

In [6]:
#文本标准化
def clean_text(s: pd.Series) -> pd.Series:
    return (s.fillna("")
             .astype(str)
             .str.strip()
             .str.replace(r"\s+", " ", regex=True)
             .str.lower())

df["keyword_clean"] = clean_text(df["Keyword"])
df["title_clean"] = clean_text(df["Title"])

In [7]:
#类型转换
numeric_cols = [
    "Position", "Previous position", "Search Volume",
    "CPC", "Traffic", "Traffic (%)",
    "Traffic Cost", "Traffic Cost (%)",
    "Competition", "Number of Results",
    "Keyword Difficulty"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["Last Seen"] = pd.to_datetime(df["Last Seen"], errors="coerce")

In [8]:
# 百分比量纲统一
for col in ["Traffic (%)", "Traffic Cost (%)"]:
    if col in df.columns:
        ratio = (df[col].dropna() <= 1).mean()
        if ratio > 0.8:
            print(f"{col} 检测为 0~1 量纲，转换为 0~100")
            df[col] = df[col] * 100

        df[col] = df[col].clip(0, 100)

Traffic (%) 检测为 0~1 量纲，转换为 0~100
Traffic Cost (%) 检测为 0~1 量纲，转换为 0~100


In [9]:
#异常处理
if "Competition" in df.columns:
    df["Competition"] = df["Competition"].clip(0, 1)

if "Position" in df.columns:
    df.loc[df["Position"] <= 0, "Position"] = np.nan

In [10]:
#去重
before = len(df)
df = df.drop_duplicates(subset=["keyword_clean", "title_clean"])
after = len(df)

print("去重数量:", before - after)

去重数量: 11


In [11]:
#生成 Keyword 维表
keyword_table = (
    df[["keyword_clean", "Keyword"]]
    .drop_duplicates("keyword_clean")
    .sort_values("keyword_clean")
    .reset_index(drop=True)
)

keyword_table["keyword_key"] = range(1, len(keyword_table) + 1)

In [12]:
#合并 keyword_key 回主表
df = df.merge(
    keyword_table[["keyword_clean", "keyword_key"]],
    on="keyword_clean",
    how="left"
)

assert df["keyword_key"].isna().sum() == 0

In [13]:
#生成分类字段 Keyword ID
def classify_keyword_id(k: str):
    k = str(k).lower()
    if any(x in k for x in ["scrum", "csm", "psm"]): return 1
    if any(x in k for x in ["aws", "devops"]): return 2
    if any(x in k for x in ["pmp", "project management"]): return 3
    if "itil" in k: return 4
    return 99

df["Keyword ID"] = df["keyword_clean"].apply(classify_keyword_id)

In [14]:
#生成 search_volume.csv
search_volume = (
    df[["keyword_key", "Search Volume"]]
    .drop_duplicates("keyword_key")
)

search_volume.to_csv("search_volume.csv", index=False)

In [15]:
#生成 keyword_difficulty.csv
keyword_difficulty = (
    df[["keyword_key", "Keyword Difficulty"]]
    .drop_duplicates("keyword_key")
)

keyword_difficulty.to_csv("keyword_difficulty.csv", index=False)

In [16]:
#生成 keyword.csv
keyword_table[["keyword_key", "Keyword"]].to_csv("keyword.csv", index=False)

In [17]:
#生成 website_traffic_data.csv
fact_cols = [
    "Title", "Keyword", "keyword_key", "Keyword ID",
    "Position", "Previous position", "Last Seen",
    "Search Volume", "CPC", "Traffic", "Traffic (%)",
    "Traffic Cost", "Traffic Cost (%)",
    "Competition", "Number of Results",
    "Keyword Difficulty"
]

website_traffic_data = df[[c for c in fact_cols if c in df.columns]].copy()

website_traffic_data.to_csv(
    "website_traffic_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("文件生成完成")

文件生成完成
